## GPU ENGINE
[Rapids AI open beta Polars post](https://developer.nvidia.com/blog/polars-gpu-engine-powered-by-rapids-cudf-now-available-in-open-beta)

[Google Colab notebook demo from Rapids AI](https://colab.research.google.com/github/rapidsai-community/showcase/blob/main/accelerated_data_processing_examples/polars_gpu_engine_demo.ipynb?utm_source=nvidia+mktg&utm_medium=web&utm_campaign=polars+launch)

In [2]:
## Only uncomment and run in colab or kaggle to install packages
# from IPython.display import clear_output
# !pip install uv
# !uv pip install "polars[gpu]>=1.27.0" --extra-index-url=https://pypi.nvidia.com
# clear_output()

In [1]:
import polars as pl
zone_schema = {"LocationID": pl.Int32, "Borough": pl.String, "Zone": pl.String, "service_zone": pl.String}

# Change path if using colab or kaggle
df_zones = pl.scan_csv("data/nyc-taxi/taxi_zone_lookup.csv", schema=zone_schema)

df = (
    pl.scan_parquet("data/nyc-taxi/yellow.parquet")
    .join(df_zones, left_on="PULocationID", right_on="LocationID")
    .rename({"Borough": "PU_Borough", "Zone": "PU_Zone", "service_zone": "PU_service_zone"})
    .join(df_zones, left_on="DOLocationID", right_on="LocationID")
    .rename({"Borough": "DO_Borough", "Zone": "DO_Zone", "service_zone": "DO_service_zone"})
)
df_green = (
    pl.scan_parquet("data/nyc-taxi/green.parquet")
    .join(df_zones, left_on="PULocationID", right_on="LocationID")
    .rename({"Borough": "PU_Borough", "Zone": "PU_Zone", "service_zone": "PU_service_zone"})
    .join(df_zones, left_on="DOLocationID", right_on="LocationID")
    .rename({"Borough": "DO_Borough", "Zone": "DO_Zone", "service_zone": "DO_service_zone"})
)

In [ ]:
df.head().collect(engine="streaming")

## Streaming Engine

In [ ]:
%%time
df.group_by("trip_distance").agg(
    pl.len().alias("trip_count"),
    pl.sum("fare_amount").alias("total_revenue"),
    pl.mean("fare_amount").alias("avg_fare"),
    pl.mean("trip_distance").alias("avg_distance"),
    pl.mean("tip_amount").alias("avg_tip"),
).sort("trip_distance").head().collect(engine="streaming")

## GPU in memory

In [12]:
gpu_in_mem_engine = pl.GPUEngine(device=0, raise_on_fail=True)

In [ ]:
%%time
# This will fail if GPU memory is not large enough
df.group_by("trip_distance").agg(
    pl.len().alias("trip_count"),
    pl.sum("fare_amount").alias("total_revenue"),
    pl.mean("fare_amount").alias("avg_fare"),
    pl.mean("trip_distance").alias("avg_distance"),
    pl.mean("tip_amount").alias("avg_tip"),
).sort("trip_distance").head().collect(engine=gpu_in_mem_engine)

In [ ]:
%%time
res, prof = (
    df_green.group_by("trip_distance")
    .agg(
        pl.len().alias("trip_count"),
        pl.sum("fare_amount").alias("total_revenue"),
        pl.mean("fare_amount").alias("avg_fare"),
        pl.mean("trip_distance").alias("avg_distance"),
        pl.mean("tip_amount").alias("avg_tip"),
    )
    .sort("trip_distance")
    .head()
    .profile(engine="streaming")
)

In [ ]:
prof.with_columns(((pl.col("end") - pl.col("start")) / pl.col("end").max() * 100).alias("pct_time_spent"))

## GPU in memory. More complex queries

In [ ]:
%%time
df.group_by(["trip_distance", "DO_Borough", "PU_Borough"]).agg(
    pl.len().alias("trip_count"),
    pl.sum("fare_amount").alias("total_revenue"),
    pl.mean("fare_amount").alias("avg_fare"),
    pl.mean("trip_distance").alias("avg_distance"),
    pl.mean("tip_amount").alias("avg_tip"),
).sort(["trip_distance", "DO_Borough", "PU_Borough"]).head().collect(engine="streaming")

In [ ]:
%%time
df.group_by(["trip_distance", "DO_Borough", "PU_Borough"]).agg(
    pl.len().alias("trip_count"),
    pl.sum("fare_amount").alias("total_revenue"),
    pl.mean("fare_amount").alias("avg_fare"),
    pl.mean("trip_distance").alias("avg_distance"),
    pl.mean("tip_amount").alias("avg_tip"),
).sort(["trip_distance", "DO_Borough", "PU_Borough"]).head().collect(engine=gpu_in_mem_engine)

## UVM for single GPU
[UVM, data larger than GPU memory post (NVIDIA post)](https://developer.nvidia.com/blog/how-to-work-with-data-exceeding-vram-in-the-polars-gpu-engine/)

[UVM, data larger than GPU memory post (Polars post)](https://pola.rs/posts/uvm-larger-than-ram-gpu/)


In [9]:
%%time
df.group_by(["trip_distance", "DO_Borough", "PU_Borough", "total_amount"]).agg(
    pl.len().alias("trip_count"),
    pl.sum("fare_amount").alias("total_revenue"),
    pl.mean("fare_amount").alias("avg_fare"),
    pl.mean("trip_distance").alias("avg_distance"),
    pl.mean("tip_amount").alias("avg_tip"),
).sort(["trip_distance", "DO_Borough", "PU_Borough", "total_amount"]).head().collect(
    engine=gpu_in_mem_engine
)  # RuntimeError: parallel_for failed: cudaErrorMemoryAllocation: out of memory

CPU times: user 0 ns, sys: 108 μs, total: 108 μs
Wall time: 109 μs


NameError: name 'gpu_in_mem_engine' is not defined

In [7]:
# Need to reset all variables to free GPU correctly and import again polars import and df definition at the top of the notebook
%restart

UsageError: Line magic function `%restart` not found.


In [10]:
import rmm

mr = rmm.mr.PrefetchResourceAdaptor(rmm.mr.PoolMemoryResource(rmm.mr.ManagedMemoryResource()))
gpu_uvm_engine = pl.GPUEngine(memory_resource=mr, raise_on_fail=True)

In [11]:
%%time
df.group_by(["trip_distance", "DO_Borough", "PU_Borough", "total_amount"]).agg(
    pl.len().alias("trip_count"),
    pl.sum("fare_amount").alias("total_revenue"),
    pl.mean("fare_amount").alias("avg_fare"),
    pl.mean("trip_distance").alias("avg_distance"),
    pl.mean("tip_amount").alias("avg_tip"),
).sort(["trip_distance", "DO_Borough", "PU_Borough", "total_amount"]).head().collect(engine=gpu_uvm_engine)

CPU times: user 8.77 s, sys: 1.08 s, total: 9.86 s
Wall time: 9.35 s


trip_distance,DO_Borough,PU_Borough,total_amount,trip_count,total_revenue,avg_fare,avg_distance,avg_tip
f32,str,str,f32,u32,f32,f32,f32,f32
-4.0840124e7,"""Queens""","""Queens""",3.8,1,2.5,2.5,-4.0840124e7,0.0
-4.0095532e7,"""Manhattan""","""Manhattan""",1.3,1,0.0,0.0,-4.0095532e7,1.0
-3.4467e6,"""Queens""","""Manhattan""",13.16,1,2.5,2.5,-3.4467e6,0.0
-186318.40625,"""Queens""","""Queens""",3.3,1,2.5,2.5,-186318.40625,0.0
-37264.53125,"""Brooklyn""","""Queens""",32.110001,1,25.809999,25.809999,-37264.53125,0.0


In [ ]:
# Need to reset all variables to free GPU correctly and import again polars import and df definition at the top of the notebook
# %reset -f 

## GPU Streaming
[Polars GPU Streaming NVIDIA blog post](https://developer.nvidia.com/blog/rapids-adds-gpu-polars-streaming-a-unified-gnn-api-and-zero-code-ml-speedups)

In [2]:
gpu_streaming_engine = pl.GPUEngine(executor="streaming", raise_on_fail=True)

In [4]:
%%time
df.group_by(["trip_distance", "DO_Borough", "PU_Borough", "total_amount"]).agg(
    pl.len().alias("trip_count"),
    pl.sum("fare_amount").alias("total_revenue"),
    pl.mean("fare_amount").alias("avg_fare"),
    pl.mean("trip_distance").alias("avg_distance"),
    pl.mean("tip_amount").alias("avg_tip"),
).sort(["trip_distance", "DO_Borough", "PU_Borough", "total_amount"]).head().collect(engine=gpu_streaming_engine)

CPU times: user 11.4 s, sys: 11.2 s, total: 22.7 s
Wall time: 19.4 s


RuntimeError: parallel_for failed: cudaErrorUnknown: unknown error

In [ ]:
## GPU Streaming